In [2]:
import psutil
print(psutil.virtual_memory())

import pandas as pd
import torch
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.enabled = True
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
import os
import gc

# Pulizia preventiva della memoria
gc.collect()
torch.cuda.empty_cache()
print("Librerie installate e cache pulita.")

svmem(total=33662472192, available=32224661504, percent=4.3, used=958091264, free=29533954048, active=1180213248, inactive=2391154688, buffers=761102336, cached=2409324544, shared=2433024, slab=286355456)
Librerie installate e cache pulita.


In [ ]:
# 1. CARICAMENTO DATI
INPUT_FILE = "/kaggle/input/datajokes/jokes_with_scores.csv"  

if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(f"File {INPUT_FILE} non trovato.")

print(f"Caricamento dati da {INPUT_FILE}...")
df = pd.read_csv(INPUT_FILE)

# Controllo colonna
if 'Total_Score' not in df.columns:
    raise ValueError("Manca la colonna Total_Score")

print(f"Dataset caricato: {len(df)} righe.")

Caricamento dati da /kaggle/input/datajokes/jokes_with_scores.csv...
Dataset caricato: 10000 righe.


In [ ]:
# 2. CONFIGURAZIONE DEBERTA
MODEL_NAME = "microsoft/deberta-v3-base"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 4
GRAD_ACCUMULATION = 2 # Accumuliamo i gradienti per simulare un batch da 8
EPOCHS = 3
LR = 2e-5

# Normalizzazione Score (0-10) per facilitare il training
min_s, max_s = df['Total_Score'].min(), df['Total_Score'].max()
df['Label'] = (df['Total_Score'] - min_s) / (max_s - min_s) * 10

In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class JokeDataset(Dataset):
    def __init__(self, df, tokenizer):
        col_text = 'jokeText' if 'jokeText' in df.columns else 'text'
        self.texts = df[col_text].values.astype(str)
        self.labels = df['Label'].values.astype(float)
        self.tokenizer = tokenizer
        
    def __len__(self): 
        return len(self.texts)
    
    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx], 
            truncation=True, 
            padding='max_length', 
            max_length=128, 
            return_tensors='pt'
        )
        return {
            'input_ids': enc['input_ids'].flatten(),
            'attention_mask': enc['attention_mask'].flatten(),
            'labels': torch.tensor(self.labels[idx], dtype=torch.float)
        }

# Creazione Dataset
train_df, val_df = train_test_split(df, test_size=0.1, random_state=42)

# Creazione DataLoader
train_loader = DataLoader(
    JokeDataset(train_df, tokenizer), 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    num_workers=0, 
    pin_memory=True
)

val_loader = DataLoader(
    JokeDataset(val_df, tokenizer), 
    batch_size=BATCH_SIZE, 
    num_workers=0, 
    pin_memory=True
)

print("DataLoader creati.")

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


DataLoader creati.


In [8]:
# Pulizia aggressiva prima di caricare il modello
gc.collect()
torch.cuda.empty_cache()

class DebertaJudge(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = AutoModel.from_pretrained(MODEL_NAME)
        self.bert.config.use_cache = False
        self.regressor = nn.Linear(self.bert.config.hidden_size, 1)
        
    def forward(self, input_ids, mask):
        out = self.bert(input_ids, attention_mask=mask)
        # Usa il token CLS
        return self.regressor(out.last_hidden_state[:, 0, :])

print("Caricamento modello in memoria...")
model = DebertaJudge()
model.to(DEVICE)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
criterion = nn.MSELoss()
scaler = torch.cuda.amp.GradScaler()

print("Modello caricato correttamente su GPU.")

Caricamento modello in memoria...


2026-01-16 19:53:30.636280: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768593210.831207      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768593210.887323      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768593211.367989      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768593211.368045      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768593211.368049      55 computation_placer.cc:177] computation placer alr

pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

Modello caricato correttamente su GPU.


/tmp/ipykernel_55/1155399690.py:23: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [ ]:
print("Inizio Training (Batch Size 1 - Stabile)...")

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    
    for i, batch in enumerate(train_loader):
        # Estrai dati
        ids = batch['input_ids'].flatten(1).to(DEVICE)
        mask = batch['attention_mask'].flatten(1).to(DEVICE)
        lbl = torch.tensor(batch['labels']).to(DEVICE).float().unsqueeze(1)
        
        optimizer.zero_grad()
        
        # Training
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            preds = model(ids, mask)
            loss = criterion(preds, lbl)
            loss = loss / GRAD_ACCUMULATION
        
        scaler.scale(loss).backward()
        
        if (i + 1) % GRAD_ACCUMULATION == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
        
        total_loss += loss.item() * GRAD_ACCUMULATION
        
        # Log ogni 500 step
        if i % 500 == 0 and i > 0:
            print(f"   Step {i}/{len(train_loader)}...")

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {avg_loss:.4f}")

print("Addestramento completato.")


OUTPUT_NAME = "deberta_judge.pth"
torch.save(model.state_dict(), OUTPUT_NAME)
print(f"Modello salvato come: {OUTPUT_NAME}")

🔥 Inizio Training (Batch Size 1 - Stabile)...


/tmp/ipykernel_55/3603185016.py:11: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lbl = torch.tensor(batch['labels']).to(DEVICE).float().unsqueeze(1)


   Step 500/2250...
   Step 1000/2250...
   Step 1500/2250...
   Step 2000/2250...
Epoch 1/3 - Loss: 3.4534
   Step 500/2250...
   Step 1000/2250...
   Step 1500/2250...
   Step 2000/2250...
Epoch 2/3 - Loss: 2.1256
   Step 500/2250...
   Step 1000/2250...
   Step 1500/2250...
   Step 2000/2250...
Epoch 3/3 - Loss: 1.6461
✅ Addestramento completato.
Modello salvato come: deberta_judge.pth
